In [ ]:
import numpy as np
import pandas as pd
import os
import random
from sklearn.model_selection import GridSearchCV, RepeatedKFold
from sklearn.svm import SVC
from sklearn.metrics import f1_score, recall_score
from sklearn.base import clone

# --- 1. CẤU HÌNH ĐƯỜNG DẪN ---
current_dir = os.getcwd()
# File đặc trưng âm thanh (vừa tạo)
X_path = "X_audio_mfcc.npy"
ids_path = "y_audio_ids.npy"
# File nhãn bệnh (để biết ai bị bệnh)
label_path = os.path.abspath(os.path.join(current_dir, '..', '..', 'data', 'raw_data', 'train_split_Depression_AVEC2017.csv'))
test_path = os.path.abspath(os.path.join(current_dir, '..', '..', 'data', 'raw_data', 'test_split_Depression_AVEC2017.csv'))

print("🚀 BẮT ĐẦU HUẤN LUYỆN MODEL AUDIO...")

# --- 2. LOAD DỮ LIỆU ---
if os.path.exists(X_path) and os.path.exists(ids_path):
    X_data = np.load(X_path)
    p_ids = np.load(ids_path)
    print(f"✅ Load xong đặc trưng âm thanh: {X_data.shape}")
else:
    print("❌ LỖI: Chưa có file .npy! Hãy chạy bước trích xuất đặc trưng trước.")
    exit()

# Load nhãn bệnh từ file CSV gốc (ghép cả train/dev/test lại cho chắc)
# Vì chúng ta cần nhãn cho TOÀN BỘ 187 người
train_df = pd.read_csv(label_path)
# (Để đơn giản, ta giả định file train_split chứa đủ thông tin hoặc load thêm dev/test nếu cần)
# Ở đây tôi viết hàm lấy nhãn thông minh:
def get_labels_for_ids(ids):
    # Load tất cả các file nhãn có thể
    dfs = []
    for f in ['train', 'dev', 'test']:
        p = os.path.abspath(os.path.join(current_dir, '..', '..', 'data', 'raw_data', f'{f}_split_Depression_AVEC2017.csv'))
        if os.path.exists(p):
            dfs.append(pd.read_csv(p))
    
    full_df = pd.concat(dfs)
    # Chuẩn hóa tên cột
    if 'participant_ID' in full_df.columns: full_df.rename(columns={'participant_ID': 'Participant_ID'}, inplace=True)
    
    full_df.set_index('Participant_ID', inplace=True)
    
    y_list = []
    valid_indices = []
    
    for i, pid in enumerate(ids):
        if pid in full_df.index:
            y_list.append(full_df.loc[pid, 'PHQ8_Binary']) # Hoặc PHQ_Binary
            valid_indices.append(i)
            
    return np.array(y_list), valid_indices

y_data, valid_idx = get_labels_for_ids(p_ids)
X_data = X_data[valid_idx] # Chỉ giữ lại những người có nhãn
p_ids = p_ids[valid_idx]

print(f"✅ Đã ghép nhãn xong. Dữ liệu cuối cùng: X={X_data.shape}, y={y_data.shape}")

# --- 3. CHIA TẬP TRAIN/TEST & CÂN BẰNG ---
def train_test_split_custom(X, y, ids, testfile):
    test_df = pd.read_csv(testfile)
    test_ids = test_df.iloc[:, 0].values
    
    X_train, X_test, y_train, y_test = [], [], [], []
    
    for i, pid in enumerate(ids):
        if pid in test_ids:
            X_test.append(X[i]); y_test.append(y[i])
        else:
            X_train.append(X[i]); y_train.append(y[i])
            
    return np.array(X_train), np.array(X_test), np.array(y_train), np.array(y_test)

X_train, X_test, y_train, y_test = train_test_split_custom(X_data, y_data, p_ids, test_path)

# Undersampling (Cân bằng)
def undersample(X, y):
    neg = [i for i, v in enumerate(y) if v == 0]
    pos = [i for i, v in enumerate(y) if v == 1]
    n = min(len(neg), len(pos))
    import random
    random.seed(42)
    idx = random.sample(neg, n) + random.sample(pos, n)
    random.shuffle(idx)
    return X[idx], y[idx]

X_train_bal, y_train_bal = undersample(X_train, y_train)
print(f"✅ Train shape (balanced): {X_train_bal.shape}")

# --- 4. HUẤN LUYỆN SVM ---
print("🧠 Đang train SVM Audio...")
params = [{'kernel': ['rbf'], 'gamma': [1e-3, 1e-4], 'C': [1, 10, 100]}]
# n_jobs=1 để tránh lỗi Windows
svm = GridSearchCV(SVC(), params, cv=5, scoring='f1', n_jobs=1)
svm.fit(X_train_bal, y_train_bal)

print("-" * 30)
print(f"🎧 AUDIO MODEL KẾT QUẢ:")
print(f"🏆 Best F1-Score (Train CV): {svm.best_score_:.4f}")
print(f"🔧 Best Params: {svm.best_params_}")

# Test trên tập Test thật
y_pred = svm.predict(X_test)
test_f1 = f1_score(y_test, y_pred)
print(f"🧪 F1-Score trên tập Test: {test_f1:.4f}")
print("-" * 30)

# Lưu model
import pickle
with open('audio_model.pkl', 'wb') as f:
    pickle.dump(svm.best_estimator_, f)
print("💾 Đã lưu model: audio_model.pkl")

🚀 BẮT ĐẦU HUẤN LUYỆN MODEL AUDIO...
❌ LỖI: Chưa có file .npy! Hãy chạy bước trích xuất đặc trưng trước.


NameError: name 'p_ids' is not defined

: 